In [1]:
!pip install --upgrade google-genai gradio pillow

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_core-2.46.4-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.2/984.2 kB 1.6 MB/s  0:00:001.3 MB/s eta 0:00:01
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp313-cp313-macosx_11_0_arm64.whl (2.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 1.9 MB/s  0:00:16 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 2.5 MB/s  0:00:022.8 MB/s eta 0:00:01:01
  Attempting uninstall: websockets
    Found existing installation: websockets 11.0.3
    Uninstalling websockets-11.0.3:
      Successfully uninstalled websockets-11.0.3
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pillow
    Found existing installation: pillow 10

In [5]:
import os
import getpass

# Enter your Gemini API Key securely when prompted
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API Key: ")

In [7]:
import os
import gradio as gr
from PIL import Image
from google import genai
from google.genai import types

# 1. Initialize the Google Gen AI Client
# The SDK automatically detects the GEMINI_API_KEY environment variable.
client = genai.Client()

# Define the models we will use
MULTIMODAL_MODEL = "gemini-2.5-flash"
IMAGEN_MODEL = "imagen-3.0-generate-002"

# 2. Chatbot core processing logic
def respond(message, history):
    """
    Processes incoming text/image messages, manages conversational flow, and decides whether
    to run a text-based analysis or trigger image generation.
    """
    # Extract structural text prompt and any attached image files from Gradio's multi-modal input
    text_prompt = message.get("text", "").strip()
    attached_files = message.get("files", [])

    # Check if the user is explicitly requesting an image generation
    image_trigger_words = ["generate an image", "create a picture", "draw", "generate image of", "create an image"]
    is_image_generation_request = any(word in text_prompt.lower() for word in image_trigger_words)

    # -------------------------------------------------------------
    # ROUTE A: Image Generation via Imagen 3
    # -------------------------------------------------------------
    if is_image_generation_request:
        try:
            yield "🎨 Processing your prompt and drawing the image... Please wait."
            
            # Request image generation from the specialized Imagen model
            result = client.models.generate_images(
                model=IMAGEN_MODEL,
                prompt=text_prompt,
                config=types.GenerateImagesConfig(
                    number_of_images=1,
                    output_mime_type="image/jpeg",
                    aspect_ratio="1:1"
                )
            )
            
            # Extract generated image bytes and save it locally
            generated_img_data = result.generated_images[0]
            output_image_path = "generated_output.jpg"
            with open(output_image_path, "wb") as f:
                f.write(generated_img_data.image.image_bytes)
            
            # Return both a confirmation text and the generated image file back to the UI
            yield {"text": f"Here is the image you requested for: '{text_prompt}'", "files": [output_image_path]}
            return
        except Exception as e:
            yield f"❌ Error generating image: {str(e)}"
            return

    # -------------------------------------------------------------
    # ROUTE B: Multimodal Chat (Text & Image Inputs) via Gemini 2.5
    # -------------------------------------------------------------
    contents = []
    
    # Reconstruct past conversation history context for Gemini
    for user_turn, assistant_turn in history:
        # User history parsing
        if isinstance(user_turn, dict):
            if user_turn.get("text"):
                contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_turn["text"])]))
            if user_turn.get("files"):
                for f_path in user_turn["files"]:
                    img = Image.open(f_path)
                    contents.append(types.Content(role="user", parts=[img]))
        elif isinstance(user_turn, str):
            contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_turn)]))

        # Assistant history parsing
        if isinstance(assistant_turn, dict):
            if assistant_turn.get("text"):
                contents.append(types.Content(role="model", parts=[types.Part.from_text(text=assistant_turn["text"])]))
        elif isinstance(assistant_turn, str):
            contents.append(types.Content(role="model", parts=[types.Part.from_text(text=assistant_turn)]))

    # Append the current active user turn
    current_parts = []
    if text_prompt:
        current_parts.append(types.Part.from_text(text=text_prompt))
    if attached_files:
        for file_info in attached_files:
            file_path = file_info if isinstance(file_info, str) else file_info.get("path")
            if file_path:
                img = Image.open(file_path)
                current_parts.append(img)
                
    contents.append(types.Content(role="user", parts=current_parts))

    try:
        # Stream the conversational response from Gemini
        response_stream = client.models.generate_content_stream(model=MULTIMODAL_MODEL, contents=contents)
        full_response = ""
        for chunk in response_stream:
            full_response += chunk.text
            yield full_response
    except Exception as e:
        yield f"❌ Error: {str(e)}"

# 3. Design and launch the web interface
with gr.Blocks() as demo:
    gr.Markdown("# 🤖 Next-Gen Gemini Multi-Modal Chatbot")
    gr.Markdown(
        "Upload images and ask questions about them, chat normally, or type "
        "`'Generate an image of a futuristic city'` to test multi-modal text-to-image workflows!"
    )
    
    # Base configuration compliant with Gradio 6.0
    chatbot = gr.Chatbot(label="Gemini AI Conversation")
    
    # Enable multi-modal inputs (text and files simultaneously)
    chat_input = gr.MultimodalTextbox(
        interactive=True, 
        file_count="multiple", 
        placeholder="Type a message or upload an image...", 
        show_label=False
    )
    
    # Link input event to our custom handler function
    chat_input.submit(respond, inputs=[chat_input, chatbot], outputs=[chat_input, chatbot])

# Pass the theme parameter inside launch() as required by Gradio 6.0+
demo.queue().launch(theme=gr.themes.Soft(), inline=True, share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
